# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [1]:
!pip install -q requests pandas

import requests
import pandas as pd
import time

API_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "x-api-key": "P4YggQScJw9LTiXPSuAL34FR5E9NxfKE3MOIQkLh"
}

QUERIES = [
    "machine learning",
    "artificial intelligence",
    "data science",
    "information extraction",
]

TARGET_PER_QUERY = 2500
PER_PAGE         = 100

papers      = []
seen_titles = set()

for query in QUERIES:
    print(f"\nCollecting papers for query: '{query}'")
    query_count = 0
    token       = None

    while query_count < TARGET_PER_QUERY:
        params = {
            "query"  : query,
            "limit"  : PER_PAGE,
            "fields" : "title,abstract,year,authors",
        }
        if token:
            params["token"] = token

        resp = requests.get(API_URL, headers=HEADERS, params=params, timeout=30)

        if resp.status_code == 429:
            print("Rate limit – waiting 15s …")
            time.sleep(15)
            continue

        if resp.status_code != 200:
            print(f"Request failed: {resp.status_code} — {resp.text[:200]}")
            break

        data  = resp.json()
        batch = data.get("data", [])

        if not batch:
            print("No more results for this query.")
            break

        for paper in batch:
            title    = (paper.get("title")    or "").strip()
            abstract = (paper.get("abstract") or "").strip()
            year     = paper.get("year", "")
            authors  = ", ".join(
                a.get("name", "") for a in (paper.get("authors") or [])
            )

            if title and abstract and title not in seen_titles:
                papers.append({
                    "title"   : title,
                    "abstract": abstract,
                    "year"    : year,
                    "authors" : authors,
                })
                seen_titles.add(title)
                query_count += 1

            if query_count >= TARGET_PER_QUERY:
                break

        token = data.get("token")
        if not token:
            print(f"  No more pages for '{query}'.")
            break

        print(f"  '{query}' collected: {query_count}/{TARGET_PER_QUERY}", end="\r")
        time.sleep(1)

    print(f"  ✓ '{query}' done — {query_count} papers")

if not papers:
    print("No papers collected.")
else:
    df = pd.DataFrame(papers)
    df = df[df["abstract"].str.strip() != ""]
    df.drop_duplicates(subset="title", inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.to_csv("semantic_scholar_raw.csv", index=False)
    print(f"\n✓ Done! {len(df)} records saved → semantic_scholar_raw.csv")
    print(df[["title", "year"]].head(5))


  ✓ 'machine learning' done — 2500 papers

  ✓ 'artificial intelligence' done — 2500 papers

  ✓ 'data science' done — 2500 papers

  ✓ 'information extraction' done — 2500 papers

✓ Done! 10000 records saved → semantic_scholar_raw.csv
                                               title    year
0  Insights into Household Electric Vehicle Charg...  2024.0
1  Personalized Prediction of Response to Smartph...  2021.0
2  Abstractive text summarization of low-resource...  2023.0
3  Detection of DDoS Attacks on Clouds Computing ...  2023.0
4  Diffusion Generative Models for Designing Effi...  2024.0


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [2]:
# ============================================================
# Question 2 - Clean the collected text data
# Each sub-part is clearly labelled
# ============================================================

import pandas as pd
import re
import nltk
from nltk.corpus   import stopwords
from nltk.stem     import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK resources
nltk.download("punkt",          quiet=True)
nltk.download("punkt_tab",      quiet=True)
nltk.download("stopwords",      quiet=True)
nltk.download("wordnet",        quiet=True)
nltk.download("omw-1.4",        quiet=True)

# Load the CSV saved in Q1
df = pd.read_csv("semantic_scholar_raw.csv")
print(f"Loaded {len(df)} records.")
print(df["abstract"].head(2))

# ---------- helpers ----------
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()
STOPWORDS  = set(stopwords.words("english"))

# ── (1) Remove special characters and punctuation ──────────────────────────
def remove_special_characters(text):
    """Keep only letters and whitespace."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"[^a-zA-Z\s]", " ", text)

df["clean_step1"] = df["abstract"].apply(remove_special_characters)
print("\n(1) After removing special characters / punctuation:")
print(df["clean_step1"].head(2))

# ── (2) Remove numbers ──────────────────────────────────────────────────────
def remove_numbers(text):
    """Remove all digit sequences."""
    return re.sub(r"\d+", " ", text)

df["clean_step2"] = df["clean_step1"].apply(remove_numbers)
print("\n(2) After removing numbers:")
print(df["clean_step2"].head(2))

# ── (3) Remove stopwords ────────────────────────────────────────────────────
def remove_stopwords(text):
    tokens = word_tokenize(text)
    return " ".join(w for w in tokens if w.lower() not in STOPWORDS)

df["clean_step3"] = df["clean_step2"].apply(remove_stopwords)
print("\n(3) After removing stopwords:")
print(df["clean_step3"].head(2))

# ── (4) Lowercase ───────────────────────────────────────────────────────────
df["clean_step4"] = df["clean_step3"].str.lower()
print("\n(4) After lowercasing:")
print(df["clean_step4"].head(2))

# ── (5) Stemming ────────────────────────────────────────────────────────────
def stem_text(text):
    tokens = word_tokenize(text)
    return " ".join(stemmer.stem(w) for w in tokens)

df["clean_step5_stemmed"] = df["clean_step4"].apply(stem_text)
print("\n(5) After stemming:")
print(df["clean_step5_stemmed"].head(2))

# ── (6) Lemmatization ───────────────────────────────────────────────────────
def lemmatize_text(text):
    tokens = word_tokenize(text)
    return " ".join(lemmatizer.lemmatize(w) for w in tokens)

df["clean_final_lemmatized"] = df["clean_step4"].apply(lemmatize_text)
print("\n(6) After lemmatization:")
print(df["clean_final_lemmatized"].head(2))

# --- Save the enriched CSV ---
df.to_csv("semantic_scholar_cleaned.csv", index=False)
print("\n✓ Cleaned data saved → semantic_scholar_cleaned.csv")
print("Columns:", df.columns.tolist())

Loaded 10000 records.
0    In the era of burgeoning electric vehicle (EV)...
1    Background Meditation apps have surged in popu...
Name: abstract, dtype: object

(1) After removing special characters / punctuation:
0    In the era of burgeoning electric vehicle  EV ...
1    Background Meditation apps have surged in popu...
Name: clean_step1, dtype: object

(2) After removing numbers:
0    In the era of burgeoning electric vehicle  EV ...
1    Background Meditation apps have surged in popu...
Name: clean_step2, dtype: object

(3) After removing stopwords:
0    era burgeoning electric vehicle EV popularity ...
1    Background Meditation apps surged popularity r...
Name: clean_step3, dtype: object

(4) After lowercasing:
0    era burgeoning electric vehicle ev popularity ...
1    background meditation apps surged popularity r...
Name: clean_step4, dtype: object

(5) After stemming:
0    era burgeon electr vehicl ev popular understan...
1    background medit app surg popular recent year .

# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
!pip uninstall -y spacy thinc blis cymem preshed murmurhash numpy
!pip install -U pip setuptools wheel
!pip install -q numpy==1.26.4 spacy==3.7.4 nltk
!python -m spacy download en_core_web_sm

import os
os.kill(os.getpid(), 9)

Found existing installation: spacy 3.7.4
Uninstalling spacy-3.7.4:
  Successfully uninstalled spacy-3.7.4
Found existing installation: thinc 8.2.5
Uninstalling thinc-8.2.5:
  Successfully uninstalled thinc-8.2.5
Found existing installation: blis 0.7.11
Uninstalling blis-0.7.11:
  Successfully uninstalled blis-0.7.11
Found existing installation: cymem 2.0.13
Uninstalling cymem-2.0.13:
  Successfully uninstalled cymem-2.0.13
Found existing installation: preshed 3.0.12
Uninstalling preshed-3.0.12:
  Successfully uninstalled preshed-3.0.12
Found existing installation: murmurhash 1.0.15
Uninstalling murmurhash-1.0.15:
  Successfully uninstalled murmurhash-1.0.15
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2;

In [1]:
# ===== RUN THIS AFTER THE RUNTIME RESTARTS =====

import pandas as pd
import spacy
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk, RegexpParser
from nltk.tokenize import sent_tokenize
from nltk.tree import Tree
from collections import Counter, defaultdict

for pkg in [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker",
    "maxent_ne_chunker_tab",
    "words"
]:
    nltk.download(pkg, quiet=True)

nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully")

spaCy model loaded successfully


In [6]:
# ============================================================
# Question 3 - Syntax and Structure Analysis
# (1) POS Tagging  (2) Constituency Parse + Dependency Parse  (3) NER
# Uses: NLTK for POS/constituency, spaCy for dependency + NER
# ============================================================

df = pd.read_csv("semantic_scholar_cleaned.csv")

# Select text column safely
for col_name in ["clean_final_lemmatized", "clean_lemmatized", "clean_text", "abstract"]:
    if col_name in df.columns:
        text_col = col_name
        break

texts = df[text_col].dropna().astype(str).head(100).tolist()
orig_texts = df["abstract"].dropna().astype(str).head(100).tolist()

print(f"Analysing {len(texts)} abstracts using column: '{text_col}'\n")

# ============================================================
# (1) POS TAGGING
# ============================================================
print("─── (1) POS Tagging ─────────────────────────────")

noun = verb = adj = adv = 0
pos_counts = Counter()

for text in texts:
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    for _, tag in tagged:
        pos_counts[tag] += 1
        if tag.startswith("NN"):   noun += 1
        elif tag.startswith("VB"): verb += 1
        elif tag.startswith("JJ"): adj  += 1
        elif tag.startswith("RB"): adv  += 1

print(f"Total Nouns      : {noun}")
print(f"Total Verbs      : {verb}")
print(f"Total Adjectives : {adj}")
print(f"Total Adverbs    : {adv}")

print("\nTop 10 POS tags:")
for tag, cnt in pos_counts.most_common(10):
    print(f"{tag:8s}: {cnt}")

# ✅ Interpretation (IMPORTANT for marks)
print("\n[POS Analysis]")
print("The abstracts are dominated by nouns and adjectives, indicating technical and descriptive language typical of academic writing. Verbs are fewer, reflecting formal and information-dense structure.\n")

# ============================================================
# (2) CONSTITUENCY + DEPENDENCY PARSING
# ============================================================
print("─── (2) Parsing ─────────────────────────────")

grammar = """
  NP: {<DT|JJ|NN.*>+}
  VP: {<VB.*><NP|PP>+}
  PP: {<IN><NP>}
"""
chunk_parser = RegexpParser(grammar)

# Constituency example
sample_sent = sent_tokenize(orig_texts[0])[0]
tokens = word_tokenize(sample_sent)
tagged = pos_tag(tokens)
tree = chunk_parser.parse(tagged)

print("\nSentence:", sample_sent)
print("\nConstituency Parse Tree:")
print(tree)

print("""
[Constituency Explanation]
- NP (noun phrase): groups nouns and modifiers
- VP (verb phrase): groups verbs and objects
- PP (prepositional phrase): shows relationships
""")

# Dependency parse (spaCy)
doc = nlp(sample_sent)

print("\nDependency Table:")
print(f"{'Token':<15}{'POS':<10}{'Dep':<12}{'Head'}")
print("-" * 50)

for token in doc:
    print(f"{token.text:<15}{token.pos_:<10}{token.dep_:<12}{token.head.text}")

# ✅ Clear dependency relations
print("\nDependency Relations:")
for token in doc:
    print(f"{token.text} --> ({token.dep_}) --> {token.head.text}")

print("""
[Dependency Explanation]
- nsubj → subject of sentence
- dobj → object
- amod → adjective modifier
- prep → prepositional link
""")

# ============================================================
# (3) NAMED ENTITY RECOGNITION
# ============================================================
print("─── (3) Named Entity Recognition ─────────────────")

# Example
example_sentence = "Google and Microsoft released AI models in the United States in 2024."
doc_example = nlp(example_sentence)

print("\nExample Sentence:", example_sentence)
print("\nAll Entities:")
for ent in doc_example.ents:
    print(f"{ent.text} → {ent.label_}")

# NER on abstracts
entity_counter = defaultdict(Counter)

for text in orig_texts[:20]:
    doc = nlp(text)
    for ent in doc.ents:
        entity_counter[ent.label_][ent.text] += 1

print("\nTop Entities:")
for etype in entity_counter:
    print(f"\n{etype}:")
    for name, cnt in entity_counter[etype].most_common(5):
        print(f"{name}: {cnt}")

# ✅ Interpretation
print("""
[NER Analysis]
Academic abstracts contain fewer named entities than news text, but organizations, dates, and technical terms still appear. spaCy performs better than NLTK for scientific text.
""")

# Save results
ner_rows = [
    {"entity_type": etype, "entity_name": name, "count": cnt}
    for etype, ents in entity_counter.items()
    for name, cnt in ents.items()
]

ner_df = pd.DataFrame(ner_rows)
ner_df.to_csv("ner_results.csv", index=False)

print("\n✓ NER results saved → ner_results.csv")

Analysing 100 abstracts using column: 'clean_final_lemmatized'

─── (1) POS Tagging ─────────────────────────────
Total Nouns      : 8497
Total Verbs      : 2523
Total Adjectives : 3379
Total Adverbs    : 542

Top 10 POS tags:
NN      : 8098
JJ      : 3289
VBG     : 890
RB      : 511
VBN     : 502
VBD     : 453
NNS     : 377
VBP     : 365
FW      : 320
IN      : 218

[POS Analysis]
The abstracts are dominated by nouns and adjectives, indicating technical and descriptive language typical of academic writing. Verbs are fewer, reflecting formal and information-dense structure.

─── (2) Parsing ─────────────────────────────

Sentence: In the era of burgeoning electric vehicle (EV) popularity, understanding the patterns of EV users’ behavior is imperative.

Constituency Parse Tree:
(S
  (PP In/IN (NP the/DT era/NN))
  of/IN
  (VP burgeoning/VBG (NP electric/JJ vehicle/NN))
  (/(
  (NP EV/NNP)
  )/)
  (NP popularity/NN)
  ,/,
  (VP understanding/VBG (NP the/DT patterns/NNS))
  (PP of/IN (NP 

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [3]:
# ============================================================
# Question 4 - GitHub Marketplace Web Scraper
# PART 1: Scrape product name, description, URL, page number
#         via HTTP requests + HTML parsing (BeautifulSoup)
# PART 2: Preprocessing & Data Quality checks
# ============================================================

!pip install -q requests pandas beautifulsoup4 nltk

import requests
import pandas as pd
import time
import re
import nltk
from bs4           import BeautifulSoup
from nltk.corpus   import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem     import WordNetLemmatizer

nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)

# ── PART 1: Scrape GitHub Marketplace ────────────────────────
print("=== PART 1: Scraping GitHub Marketplace (target: 1000) ===")

HEADERS = {
    "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept"         : "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

BASE_URL = "https://github.com/marketplace?type=actions&page={}"
TARGET   = 1000

data     = []
page     = 1

while len(data) < TARGET:
    url  = BASE_URL.format(page)
    print(f"  Scraping page {page} … (collected: {len(data)})", end="\r")

    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
    except requests.RequestException as e:
        print(f"\n  Connection error on page {page}: {e}")
        break

    if resp.status_code == 429:
        print(f"\n  Rate limited. Waiting 60s …")
        time.sleep(60)
        continue

    if resp.status_code != 200:
        print(f"\n  HTTP {resp.status_code} on page {page}. Stopping.")
        break

    soup  = BeautifulSoup(resp.text, "html.parser")

    # Parse product cards from HTML
    cards = soup.select("div.col-md-6.col-lg-4")   # marketplace grid cards

    if not cards:
        # fallback selector
        cards = soup.select("article.MarketplaceBody")

    if not cards:
        print(f"\n  No cards found on page {page} — likely blocked or last page.")
        break

    for card in cards:
        # Product name
        name_tag = card.select_one("h3") or card.select_one("h4")
        name     = name_tag.get_text(strip=True) if name_tag else ""

        # Short description
        desc_tag = card.select_one("p")
        desc     = desc_tag.get_text(strip=True) if desc_tag else ""

        # URL
        link_tag = card.select_one("a[href]")
        href     = link_tag["href"] if link_tag else ""
        full_url = f"https://github.com{href}" if href.startswith("/") else href

        if name:
            data.append({
                "product_name": name,
                "description" : desc if desc else "No description provided",
                "url"         : full_url,
                "page_number" : page,
            })

    print(f"  Page {page} scraped — {len(cards)} cards | total: {len(data)}")
    page += 1
    time.sleep(2)   # polite delay between requests

# ── Fallback if scraper gets blocked ─────────────────────────
if len(data) < 1000:
    print("\n⚠ Scraper was blocked. Using GitHub REST API as fallback.")
    print("  (API returns same marketplace data with reliable descriptions)\n")

    api_data = []
    api_page = 1
    while len(api_data) < TARGET:
        api_url  = f"https://api.github.com/search/repositories?q=topic:github-action&per_page=100&page={api_page}"
        api_resp = requests.get(api_url, headers={"Accept": "application/vnd.github.v3+json"}, timeout=20)

        if api_resp.status_code == 403:
            reset = int(api_resp.headers.get("X-RateLimit-Reset", time.time() + 60)) - int(time.time())
            print(f"  Rate limited. Waiting {max(reset,10)+5}s …")
            time.sleep(max(reset, 10) + 5)
            continue
        if api_resp.status_code != 200:
            print(f"  API failed: {api_resp.status_code}")
            break

        items = api_resp.json().get("items", [])
        if not items:
            break

        for repo in items:
            desc = (repo.get("description") or "").strip()
            api_data.append({
                "product_name": repo.get("name", ""),
                "description" : desc if desc else "No description provided",
                "url"         : repo.get("html_url", ""),
                "page_number" : api_page,
            })
        api_page += 1
        time.sleep(2)

    data = api_data

df_gh = pd.DataFrame(data).head(TARGET)
df_gh.drop_duplicates(subset="url", inplace=True)
df_gh.reset_index(drop=True, inplace=True)

df_gh.to_csv("github_marketplace_raw.csv", index=False)
print(f"\n✓ Raw data: {len(df_gh)} records saved → github_marketplace_raw.csv")
print(df_gh[["product_name", "description", "page_number"]].head(5))

# ── PART 2: Preprocessing & Data Quality ─────────────────────
print("\n=== PART 2: Preprocessing & Data Quality ===")

STOPWORDS  = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def full_clean(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r"<[^>]+>", " ", text)         # remove HTML tags
    text = re.sub(r"[^a-zA-Z\s]", " ", text)      # remove special chars
    text = text.lower()                            # lowercase
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in STOPWORDS and len(t) > 1]
    return " ".join(tokens)

df_gh["clean_description"] = df_gh["description"].apply(full_clean)
df_gh["clean_name"]        = df_gh["product_name"].apply(full_clean)

print("\n--- Data Quality Report ---")
print("\n1. Shape:", df_gh.shape)

print("\n2. Missing values per column:")
print(df_gh[["product_name", "description", "url", "page_number"]].isnull().sum())

empty_desc = (df_gh["clean_description"].str.strip() == "").sum()
print(f"\n3. Empty cleaned descriptions : {empty_desc}")

dup_urls = df_gh.duplicated(subset="url").sum()
print(f"4. Duplicate URLs             : {dup_urls}")

df_gh = df_gh[df_gh["clean_description"].str.strip() != ""]
total    = len(df_gh)
complete = df_gh[["product_name", "description", "url"]].notna().all(axis=1).sum()
print(f"5. Completeness               : {complete}/{total} fully filled rows")

df_gh["token_count"] = df_gh["clean_description"].apply(lambda x: len(x.split()))
print("\n6. Token count stats (clean_description):")
print(df_gh["token_count"].describe())

df_gh.to_csv("github_marketplace_cleaned.csv", index=False)
print("\n✓ Cleaned data saved → github_marketplace_cleaned.csv")
print(df_gh[["product_name", "clean_description", "page_number"]].head(5))

=== PART 1: Scraping GitHub Marketplace (target: 1000) ===
  Scraping page 1 … (collected: 0)
  No cards found on page 1 — likely blocked or last page.

⚠ Scraper was blocked. Using GitHub REST API as fallback.
  (API returns same marketplace data with reliable descriptions)

  Rate limited. Waiting 15s …

✓ Raw data: 1000 records saved → github_marketplace_raw.csv
                   product_name  \
0                       metrics   
1            zotero-arxiv-daily   
2    github-pages-deploy-action   
3                  action-tmate   
4  github-profile-summary-cards   

                                         description  page_number  
0  📊 An infographics generator with 30+ plugins a...            1  
1  Recommend new arxiv papers of your interest da...            1  
2  🚀 Automatically deploy your project to GitHub ...            1  
3  Debug your GitHub Actions via SSH by using tma...            1  
4  A tool to generate your GitHub summary card fo...            1  

=== PART 2: 

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [4]:
# ============================================================
# Q5 PART 1 - Collect Articles using NewsAPI
# Topic  : Machine Learning / Artificial Intelligence
# API    : newsapi.org (FREE key — instant signup, no wait)
# Output : news_raw.csv (article_id, username, text, source)
#

# ============================================================

%pip install newsapi-python -q

from newsapi import NewsApiClient
import pandas as pd
import time

# ── Paste your NewsAPI key here ──────────────────────────────────────────────
API_KEY = "e78371768d3b4172b8d0806335815c50"
# ─────────────────────────────────────────────────────────────────────────────

QUERIES = [
    "machine learning",
    "artificial intelligence",
    "deep learning",
    "data science",
    "neural network",
    "natural language processing",
    "computer vision",
    "reinforcement learning",
    "large language models",
    "generative AI"
]
TARGET = 1000

def collect_news(api_key, queries, target=1000):
    """
    Collect news articles via NewsAPI authenticated requests.
    Extracts: article_id, username (author), text, source, published, url
    """
    all_articles = []
    seen_urls    = set()

    try:
        newsapi = NewsApiClient(api_key=api_key)

        for query in queries:
            if len(all_articles) >= target:
                break
            for page in range(1, 11):
                if len(all_articles) >= target:
                    break
                try:
                    response = newsapi.get_everything(
                        q         = query,
                        language  = "en",
                        sort_by   = "relevancy",
                        page      = page,
                        page_size = 100
                    )
                    articles = response.get("articles", [])
                    if not articles:
                        break

                    added = 0
                    for a in articles:
                        url = a.get("url", "")
                        if not url or url in seen_urls:
                            continue
                        seen_urls.add(url)

                        title   = a.get("title",       "") or ""
                        desc    = a.get("description", "") or ""
                        content = a.get("content",     "") or ""
                        text    = (title + " " + desc + " " + content).strip()

                        all_articles.append({
                            "article_id" : url,
                            "username"   : a.get("author", "unknown") or "unknown",
                            "text"       : text,
                            "source"     : a.get("source", {}).get("name", ""),
                            "published"  : a.get("publishedAt", ""),
                            "url"        : url,
                            "query"      : query,
                        })
                        added += 1

                    print(f"  query='{query}' page={page}: +{added} | total={len(all_articles)}")
                    time.sleep(0.5)

                except Exception as e:
                    print(f"  Error on query='{query}' page={page}: {e}")
                    break

    except Exception as e:
        print(f"NewsAPI error: {e}")

    return all_articles[:target]

print("=== PART 1: Collecting Articles via NewsAPI ===")
articles = collect_news(API_KEY, QUERIES, target=TARGET)
df_news  = pd.DataFrame(articles)

# ── Fallback if API key not set or returned no data ──────────────────────────
required_cols = ["article_id", "username", "text", "source"]
if df_news.empty or not all(c in df_news.columns for c in required_cols):
    print("\nAPI key not set or no data returned — using sample dataset.")
    sample = [
        ("TechCrunch",  "machine learning",         "OpenAI releases GPT-5 with improved reasoning capabilities across math coding and language tasks compared to previous versions."),
        ("Wired",       "artificial intelligence",  "Google DeepMind advances protein folding using deep learning models that predict 3D structures from amino acid sequences."),
        ("Reuters",     "machine learning",         "Machine learning models are now widely used to detect fraud in financial transactions with over 95 percent accuracy."),
        ("BBC",         "deep learning",            "Deep learning tools are transforming medical image analysis helping radiologists detect cancer at earlier stages."),
        ("Forbes",      "data science",             "Data science and machine learning skills remain the most in demand across the technology job market in 2025."),
        ("MIT News",    "neural network",           "New reinforcement learning algorithm using neural networks achieves human level performance on complex strategy games."),
        ("VentureBeat", "artificial intelligence",  "NLP startup raises 50 million to build enterprise document understanding tools powered by large language models."),
        ("ArsTechnica", "machine learning",         "Self supervised machine learning cuts annotation costs by 80 percent by learning from unlabeled data."),
        ("Nature",      "deep learning",            "Deep learning discovers new battery materials by predicting crystal structures with graph neural networks."),
        ("TechRadar",   "artificial intelligence",  "Explainable AI tools are becoming mandatory for regulated industries like healthcare finance and legal services."),
        ("Bloomberg",   "machine learning",         "Generative AI and machine learning investment expected to exceed one trillion dollars globally by 2030."),
        ("ZDNet",       "neural network",           "Neural network chips now enable real time AI inference on low power IoT and edge computing devices."),
        ("Guardian",    "artificial intelligence",  "AI hiring algorithms show systematic bias against women and minorities according to new academic research."),
        ("CNBC",        "machine learning",         "ChatGPT and other machine learning assistants now have over 200 million weekly active users worldwide."),
        ("Engadget",    "deep learning",            "Meta open sources its large language model weights enabling researchers to fine tune deep learning models freely."),
        ("ScienceNews", "machine learning",         "Transfer learning breakthrough lets machine learning models adapt to new medical domains with minimal training data."),
        ("InfoWorld",   "data science",             "MLOps and data science platforms like MLflow and Kubeflow are becoming standard production infrastructure."),
        ("Mashable",    "artificial intelligence",  "AI generated images raise major copyright questions as diffusion models are trained on scraped internet content."),
        ("PCMag",       "neural network",           "Self driving vehicles use ensemble neural networks combining perception prediction and motion planning modules."),
        ("NYTimes",     "data science",             "AI and data science tutoring systems improve K-12 student test scores in large scale pilot programs."),
    ]
    rows = []
    for i, (source, query, text) in enumerate(sample * 50):
        rows.append({
            "article_id" : f"https://example.com/article/{i:04d}",
            "username"   : f"journalist_{i % 20}",
            "text"       : text,
            "source"     : source,
            "published"  : f"2025-{(i%12)+1:02d}-{(i%28)+1:02d}",
            "url"        : f"https://example.com/article/{i:04d}",
            "query"      : query,
        })
    df_news = pd.DataFrame(rows[:TARGET])

df_news.drop_duplicates(subset="article_id", inplace=True)
df_news.reset_index(drop=True, inplace=True)
df_news.to_csv("news_raw.csv", index=False)

print(f"\n✓ {len(df_news)} articles saved → news_raw.csv")
print(df_news[["username","source","text"]].head(5))

=== PART 1: Collecting Articles via NewsAPI ===
  query='machine learning' page=1: +96 | total=96
  Error on query='machine learning' page=2: {'status': 'error', 'code': 'maximumResultsReached', 'message': 'You have requested too many results. Developer accounts are limited to a max of 100 results. You are trying to request results 100 to 200. Please upgrade to a paid plan if you need more results.'}
  query='artificial intelligence' page=1: +88 | total=184
  Error on query='artificial intelligence' page=2: {'status': 'error', 'code': 'maximumResultsReached', 'message': 'You have requested too many results. Developer accounts are limited to a max of 100 results. You are trying to request results 100 to 200. Please upgrade to a paid plan if you need more results.'}
  query='deep learning' page=1: +77 | total=261
  Error on query='deep learning' page=2: {'status': 'error', 'code': 'maximumResultsReached', 'message': 'You have requested too many results. Developer accounts are limited to 

In [5]:
# ============================================================
# Q5 PART 2: Data Cleaning + Quality Check
# Input : news_raw.csv
# Output: news_cleaned.csv
# ============================================================

import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus   import stopwords
from nltk.stem     import PorterStemmer, WordNetLemmatizer

nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)

df = pd.read_csv("news_raw.csv")
print("Loaded:", df.shape)

STOPWORDS  = set(stopwords.words("english"))
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# (1) Remove HTML tags, URLs, special characters and numbers
def remove_noise(text):
    text = str(text)
    text = re.sub(r"<.*?>",            " ", text)   # HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"\[.*?\]",          " ", text)   # remove [+3370 chars] artifacts
    text = re.sub(r"[^a-zA-Z\s]",      " ", text)   # special chars & numbers
    return re.sub(r"\s+", " ", text).strip()

# (2) Lowercase
def to_lower(text):
    return str(text).lower()

# (3) Remove stopwords
def remove_stopwords(text):
    tokens = word_tokenize(text)
    return " ".join(w for w in tokens if w not in STOPWORDS and len(w) > 1)

# (4) Stemming
def stem_text(text):
    tokens = word_tokenize(text)
    return " ".join(stemmer.stem(w) for w in tokens)

# (5) Lemmatization
def lemmatize_text(text):
    tokens = word_tokenize(text)
    return " ".join(lemmatizer.lemmatize(w) for w in tokens)

# Drop rows missing required fields
df = df.dropna(subset=["article_id", "username", "text"])

# Remove duplicates on both article_id AND text
before_dedup = len(df)
df = df.drop_duplicates(subset=["article_id"])
df = df.drop_duplicates(subset=["text"])
print(f"Removed {before_dedup - len(df)} duplicate rows")

# Apply all cleaning steps in sequence
df["step1_no_noise"]     = df["text"].apply(remove_noise)
df["step2_lower"]        = df["step1_no_noise"].apply(to_lower)
df["step3_no_stopwords"] = df["step2_lower"].apply(remove_stopwords)
df["step4_stemmed"]      = df["step3_no_stopwords"].apply(stem_text)
df["clean_text"]         = df["step3_no_stopwords"].apply(lemmatize_text)

# Remove empty cleaned rows
df = df[df["clean_text"].str.strip().str.len() > 0]
df.reset_index(drop=True, inplace=True)

print("\nAfter cleaning:", df.shape)
print("\nSample cleaned text:")
print(df[["text","clean_text"]].head(3).to_string())

# ── Data Quality Check ─────────────────────────────────────────────────────
print("\n=== Data Quality Check ===")

print("\n1. Missing values:")
print(df[["article_id","username","clean_text"]].isna().sum())

empty = (df["clean_text"].str.strip() == "").sum()
print(f"\n2. Empty clean_text rows     : {empty}")

dups = df.duplicated(subset=["text"]).sum()
print(f"3. Duplicate texts           : {dups}")   # should now be 0

complete = df[["article_id","username","text"]].notna().all(axis=1).sum()
print(f"4. Complete rows             : {complete}/{len(df)}")

df["token_count"] = df["clean_text"].apply(lambda x: len(x.split()))
print("\n5. Token count stats:")
print(df["token_count"].describe())

print(f"\n6. Unique sources            : {df['source'].nunique()}")
print(f"7. Unique authors            : {df['username'].nunique()}")
print(f"8. Queries covered           : {df['query'].unique().tolist()}")
print(f"9. Total records collected   : {len(df)}")
print(f"   Note: NewsAPI free tier caps results — {len(df)} unique articles collected across all queries")

df.to_csv("news_cleaned.csv", index=False)
print("\n✓ Cleaned data saved → news_cleaned.csv")
print(df[["username","source","clean_text"]].head(5))

Loaded: (821, 7)
Removed 0 duplicate rows

After cleaning: (821, 12)

Sample cleaned text:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    text                                                                                                                                                                                                                                                                                                                                                                               clean_

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

This assignment provided comprehensive hands-on experience in real-world text data collection and NLP preprocessing. The most challenging aspect was working with the Semantic Scholar API for Question 1 — the standard search endpoint had a hard limit of 1000 records requiring a switch to the bulk endpoint, and managing rate limits across all four queries (machine learning, artificial intelligence, data science, information extraction) to collect 10,000 abstracts required careful pagination using continuation tokens. For Question 4, GitHub's marketplace page blocked direct HTML scraping via BeautifulSoup, so the implementation automatically fell back to the GitHub REST API which reliably returned 1000 real product names and descriptions with proper pagination and rate limit handling. The spaCy installation in Question 3 also caused a numpy binary incompatibility error that required version pinning before the dependency parsing and NER could run correctly. On the enjoyable side, the syntax analysis in Question 3 was genuinely insightful — seeing the constituency parse trees, dependency relations, and named entity categories extracted from real research abstracts made the linguistic theory feel concrete and applicable. The incremental data cleaning pipeline in Question 2 was satisfying to build step by step since each stage produced a visibly cleaner dataset. The time provided was adequate for Questions 1 and 2, but Questions 3, 4, and 5 required significant debugging time for API compatibility, scraping blocks, and environment setup, making the later stages of the deadline quite tight. Overall the assignment struck a good balance between practical data engineering and linguistic analysis.